# ProteinOPD Training Demo

This notebook runs a small Colab-friendly demo of the ProteinOPD training flow.

It is intended to verify the teacher-to-student pipeline on tiny datasets. It is not a full paper-scale reproduction. For full experiments, use the repository scripts on a multi-GPU server.

Use a GPU runtime: `Runtime > Change runtime type > T4/L4/A100 GPU`.

In [ ]:
#@title Setup
repo_url = "https://github.com/THU-AI4S/ProteinOPD.git" #@param {type:"string"}
branch = "main" #@param {type:"string"}
mount_google_drive = True #@param {type:"boolean"}

import os
import subprocess
import sys
from pathlib import Path

def run(cmd, cwd=None):
    print("$", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

run(["nvidia-smi"])

repo_dir = Path("/content/ProteinOPD")
if repo_dir.exists():
    run(["git", "fetch", "origin"], cwd=repo_dir)
    run(["git", "checkout", branch], cwd=repo_dir)
    run(["git", "pull", "--ff-only"], cwd=repo_dir)
else:
    run(["git", "clone", "--branch", branch, repo_url, str(repo_dir)])

run([
    sys.executable, "-m", "pip", "install", "-q",
    "accelerate", "datasets", "huggingface_hub", "peft", "pyyaml",
    "safetensors", "sentencepiece", "tensorboard", "tokenizers", "tqdm", "transformers", "wandb"
])

if mount_google_drive:
    from google.colab import drive
    drive.mount("/content/drive")

print("Repository:", repo_dir)

In [ ]:
#@title Optional Hugging Face login
login_to_huggingface = False #@param {type:"boolean"}

if login_to_huggingface:
    from huggingface_hub import notebook_login
    notebook_login()
else:
    print("Skipping Hugging Face login. Enable this for private or gated models.")

In [ ]:
#@title Demo settings
track = "unconditional" #@param ["unconditional", "conditional"]
preference_dataset = "sol" #@param ["sol", "foldability", "thermo"]

protgpt2_model_id = "nferruz/ProtGPT2" #@param {type:"string"}
prollama_model_id = "GreatCaptainNemo/ProLLaMA" #@param {type:"string"}

num_train_samples = 64 #@param {type:"integer"}
teacher_epochs = 1 #@param {type:"integer"}
student_epochs = 1 #@param {type:"integer"}
per_device_batch_size = 1 #@param {type:"integer"}
gradient_accumulation_steps = 1 #@param {type:"integer"}
max_length = 256 #@param {type:"integer"}
max_new_tokens = 128 #@param {type:"integer"}

output_root = "/content/drive/MyDrive/ProteinOPD/outputs" #@param {type:"string"}
upload_to_hub = False #@param {type:"boolean"}
hub_repo_id = "THU-AI4S/proteinopd-demo-adapter" #@param {type:"string"}

output_root = Path(output_root)
work_dir = Path("/content/proteinopd_training_demo")
output_root.mkdir(parents=True, exist_ok=True)
work_dir.mkdir(parents=True, exist_ok=True)

print("Track:", track)
print("Dataset:", preference_dataset)
print("Outputs:", output_root)

In [ ]:
import torch
from huggingface_hub import snapshot_download

if not torch.cuda.is_available():
    raise RuntimeError("This demo requires a GPU runtime.")

precision_flag = "--bf16" if torch.cuda.is_bf16_supported() else "--fp16"
print("Using precision:", precision_flag)

def resolve_local_model(model_id_or_path):
    candidate = Path(model_id_or_path).expanduser()
    if candidate.exists():
        return str(candidate.resolve())
    print("Downloading model snapshot from Hugging Face:", model_id_or_path)
    return snapshot_download(repo_id=model_id_or_path)

local_prollama_model_path = resolve_local_model(prollama_model_id) if track == "conditional" else prollama_model_id

def dataset_path(kind, split):
    if kind == "unconditional":
        name = "train" if split == "train" else "test"
        suffix = "200" if split == "train" else "50"
        if preference_dataset == "sol":
            filename = f"{name}_sol_{suffix}.csv"
        elif preference_dataset == "foldability":
            filename = f"{name}_plddt_{suffix}.csv"
        else:
            filename = f"{name}_thermo_{suffix}.csv"
        return repo_dir / "unconditional" / "data" / preference_dataset / filename
    name = "train" if split == "train" else "test"
    suffix = "200" if split == "train" else "50"
    if preference_dataset == "sol":
        filename = f"{name}_sol_{suffix}.json"
    elif preference_dataset == "foldability":
        filename = f"{name}_plddt_{suffix}.json"
    else:
        filename = f"{name}_thermo_{suffix}.json"
    return repo_dir / "conditional" / "data" / preference_dataset / filename

train_data = dataset_path(track, "train")
test_data = dataset_path(track, "test")
print("Train data:", train_data)
print("Validation data:", test_data)

In [ ]:
import yaml

def run_unconditional_demo():
    teacher_output = output_root / "unconditional_teacher_prefix"
    teacher_config_path = work_dir / "unconditional_teacher.yaml"
    teacher_config = {
        "model_name_or_path": protgpt2_model_id,
        "tokenizer_name_or_path": protgpt2_model_id,
        "train_path": str(train_data),
        "test_path": str(test_data),
        "text_column": "sequence",
        "dataset_name": f"proteinopd_{preference_dataset}_demo",
        "output_dir": str(teacher_output),
        "log_dir": str(work_dir / "logs" / "unconditional_teacher"),
        "run_dir": str(work_dir / "runs" / "unconditional_teacher"),
        "num_virtual_tokens": 20,
        "max_length": max_length,
        "learning_rate": 0.005,
        "num_epochs": teacher_epochs,
        "batch_size": per_device_batch_size,
        "seed": 42,
        "device": "auto",
        "early_stop": False,
    }
    teacher_config_path.write_text(yaml.safe_dump(teacher_config), encoding="utf-8")
    run([sys.executable, "unconditional/teacher_construct/prefix_tuning_prot.py", "--config", str(teacher_config_path)], cwd=repo_dir)

    adapter_dirs = sorted(teacher_output.glob("prefix_*"))
    if not adapter_dirs:
        raise RuntimeError(f"No teacher adapter found under {teacher_output}")
    teacher_adapter = adapter_dirs[-1]
    print("Teacher adapter:", teacher_adapter)

    opd_teacher_config_path = work_dir / "unconditional_opd_teachers.yaml"
    opd_teacher_config = {
        "log_teacher_entropy": False,
        "teacher_backbone_path": protgpt2_model_id,
        "teachers": [
            {"name": "teacher_1", "adapter_path": str(teacher_adapter), "weight": 1.0, "temperature": 1.0}
        ],
    }
    opd_teacher_config_path.write_text(yaml.safe_dump(opd_teacher_config), encoding="utf-8")

    student_output = output_root / "unconditional_student_opd"
    cmd = [
        sys.executable, "unconditional/proteinopd/protein_opd_train.py",
        "--num_train_samples", str(num_train_samples),
        "--student_model_name_or_path", protgpt2_model_id,
        "--teacher_config_path", str(opd_teacher_config_path),
        "--prompt_mode", "unconditional",
        "--max_new_tokens", str(max_new_tokens),
        "--temperature", "1.0",
        "--repetition_penalty", "1.2",
        "--top_p", "0.95",
        "--top_k", "500",
        "--beta", "0.5",
        "--useloss", "jsd",
        "--student_tune_mode", "lora",
        "--lora_r", "8",
        "--lora_alpha", "16",
        "--lora_dropout", "0.05",
        "--lora_target_modules", "c_attn", "c_proj", "c_fc",
        "--per_device_train_batch_size", str(per_device_batch_size),
        "--gradient_accumulation_steps", str(gradient_accumulation_steps),
        "--learning_rate", "2e-5",
        "--num_train_epochs", str(student_epochs),
        "--logging_steps", "1",
        "--save_steps", "50",
        "--report_to", "none",
        "--output_dir", str(student_output),
        precision_flag,
    ]
    run(cmd, cwd=repo_dir)
    return student_output

def run_conditional_demo():
    teacher_output = output_root / "conditional_teacher_lora"
    teacher_script = repo_dir / "conditional" / "teacher_construct" / "scripts" / "instruction_tune.py"
    cmd = [
        sys.executable, str(teacher_script),
        "--model_name_or_path", local_prollama_model_path,
        "--tokenizer_name_or_path", local_prollama_model_path,
        "--train_file", str(train_data),
        "--validation_file", str(test_data),
        "--per_device_train_batch_size", str(per_device_batch_size),
        "--per_device_eval_batch_size", str(per_device_batch_size),
        "--do_train", "--do_eval", "--seed", "42",
        precision_flag,
        "--num_train_epochs", str(teacher_epochs),
        "--lr_scheduler_type", "cosine",
        "--learning_rate", "2e-5",
        "--warmup_ratio", "0.05",
        "--logging_strategy", "steps",
        "--logging_steps", "1",
        "--eval_strategy", "steps",
        "--eval_steps", "20",
        "--save_strategy", "steps",
        "--save_steps", "50",
        "--save_total_limit", "2",
        "--gradient_accumulation_steps", str(gradient_accumulation_steps),
        "--preprocessing_num_workers", "1",
        "--max_seq_length", str(max_length),
        "--output_dir", str(teacher_output),
        "--report_to", "none",
        "--lora_rank", "8",
        "--lora_alpha", "16",
        "--trainable", "q_proj,v_proj,k_proj,o_proj,gate_proj,down_proj,up_proj",
        "--lora_dropout", "0.1",
        "--torch_dtype", "bfloat16" if precision_flag == "--bf16" else "float16",
        "--load_in_kbits", "16",
        "--save_safetensors", "False",
        "--gradient_checkpointing",
    ]
    run(cmd, cwd=teacher_script.parent)

    opd_config_path = work_dir / "conditional_opd.yaml"
    opd_config = {
        "student": {
            "backbone_path": local_prollama_model_path,
            "lora": {
                "r": 8,
                "alpha": 16,
                "dropout": 0.1,
                "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
            },
        },
        "teachers": {
            "backbone_path": local_prollama_model_path,
            "adapters": [{"name": "teacher_1", "adapter_path": str(teacher_output), "weight": 1.0, "temperature": 0.7}],
        },
        "prompt": {
            "instruction": "[Generate by superfamily]",
            "input": "Superfamily=<Lysozyme-like domain superfamily>",
            "sequence_regex": "Seq=<([^>]*)>",
        },
        "distill": {"loss_type": "jsd", "beta": 0.5, "top_k_loss": 0, "protein_opd": {}},
        "generation": {
            "num_train_samples": num_train_samples,
            "max_new_tokens": max_new_tokens,
            "student_temperature": 1.0,
            "repetition_penalty": 1.2,
            "top_p": 0.9,
            "top_k": 200,
            "save_generations": True,
            "generation_save_steps": 50,
        },
    }
    opd_config_path.write_text(yaml.safe_dump(opd_config), encoding="utf-8")

    student_output = output_root / "conditional_student_opd"
    cmd = [
        sys.executable, "conditional/proteinopd/prollama_opd_train.py",
        "--opd_config_path", str(opd_config_path),
        "--output_dir", str(student_output),
        "--per_device_train_batch_size", str(per_device_batch_size),
        "--gradient_accumulation_steps", str(gradient_accumulation_steps),
        "--do_train", "--seed", "42",
        precision_flag,
        "--num_train_epochs", str(student_epochs),
        "--lr_scheduler_type", "cosine",
        "--learning_rate", "2e-5",
        "--warmup_ratio", "0.1",
        "--logging_strategy", "steps",
        "--logging_steps", "1",
        "--report_to", "none",
        "--save_strategy", "steps",
        "--save_steps", "50",
        "--save_total_limit", "2",
        "--eval_strategy", "no",
        "--load_in_kbits", "16",
        "--save_safetensors", "False",
        "--gradient_checkpointing",
    ]
    run(cmd, cwd=repo_dir)
    return student_output

student_output = run_unconditional_demo() if track == "unconditional" else run_conditional_demo()
print("Student adapter output:", student_output)

In [ ]:
#@title Optional upload to Hugging Face Hub
if upload_to_hub:
    from huggingface_hub import HfApi, create_repo
    create_repo(hub_repo_id, exist_ok=True)
    api = HfApi()
    api.upload_folder(folder_path=str(student_output), repo_id=hub_repo_id, repo_type="model")
    print("Uploaded to:", hub_repo_id)
else:
    print("Skipping upload. Adapter is saved at:", student_output)